In [ ]:
!pip install cell-census

In [ ]:
!pip install scanpy

# Get example blood/immune datasets

In [1]:
import os

import anndata
import scanpy as sc
import pandas as pd
import numpy as np

from os.path import join
from cellxgene_census import download_source_h5ad

## Download raw data from CELLxGENE

List of dataset collections:
* https://cellxgene.cziscience.com/collections/ced320a1-29f3-47c1-a735-513c7084d508
* https://cellxgene.cziscience.com/collections/8f126edf-5405-4731-8374-b5ce11f53e82
* https://cellxgene.cziscience.com/collections/b0cf0afa-ec40-4d65-b570-ed4ceacc6813/private
* https://cellxgene.cziscience.com/collections/4f889ffc-d4bc-4748-905b-8eb9db47a2ed
* https://cellxgene.cziscience.com/collections/ed9185e3-5b82-40c7-9824-b2141590c7f0 (submitted as 2 subsets)
* https://cellxgene.cziscience.com/collections/436154da-bcf1-4130-9c8b-120ff9a888f2
* https://cellxgene.cziscience.com/collections/b9fc3d70-5a72-4479-a046-c2cc1ab19efc
* https://cellxgene.cziscience.com/collections/ddfad306-714d-4cc0-9985-d9072820c530
* https://cellxgene.cziscience.com/collections/7d7cabfd-1d1f-40af-96b7-26a0825a306d
* https://cellxgene.cziscience.com/collections/eb735cc9-d0a7-48fa-b255-db726bf365af
* https://cellxgene.cziscience.com/collections/e415fd00-4811-4a11-a35d-0e81b7f9eb4d (submitted as 3 subsets)
* https://cellxgene.cziscience.com/collections/dde06e0f-ab3b-46be-96a2-a8082383c4a1/private
* https://cellxgene.cziscience.com/collections/03f821b4-87be-4ff4-b65a-b5fc00061da7

In [2]:
dataset_collections = [
    # Format: (List of datasets in collection, name of the cell type annotation column to use)
    (['b0e547f0-462b-4f81-b31b-5b0a5d96f537'], 'author_cell_type'),
    (['ebc2e1ff-c8f9-466a-acf4-9d291afaf8b3'], 'cell_type_source'),
    (['ed5d841d-6346-47d4-ab2f-7119ad7e3a35'], 'celltype.l3'),
    (['de2c780c-1747-40bd-9ccf-9588ec186cee'], 'Celltype'),
    (['21d3e683-80a4-4d9b-bc89-ebb2df513dde', '30cd5311-6c09-46c9-94f1-71fe4b91813c'], 'author_cell_type'),
    (['218acb0f-9f2f-4f76-b90b-15a4b7c7f629'], 'author_cell_type'),
    (['96a3f64b-0ee9-40d8-91e9-813ce38261c9'], 'Cell.class'),
    (['01ad3cd7-3929-4654-84c0-6db05bd5fd59'], 'ct3'),
    (['c2a461b1-0c15-4047-9fcb-1f966fe55100'], 'Annotation'),
    (['3faad104-2ab8-4434-816d-474d8d2641db'], 'predicted.celltype.l2'),
    (['2a498ace-872a-4935-984b-1afa70fd9886'], 'annotation_detailed_fullNames'),
]

In [3]:
DOWNLOAD_PATH = '/mnt/dssfs02/dataset-similarity'

In [4]:
for datasets, _ in dataset_collections:
    for dataset in datasets:
        print(f'Downloading {dataset}')
        save_path = join(DOWNLOAD_PATH, f'{dataset}.h5ad')
        if not os.path.isfile(save_path):
            download_source_h5ad(dataset, to_path=save_path)


## Preprocess datasets

In [5]:
import numpy as np
from scipy.sparse import csc_matrix, csr_matrix


def streamline_count_matrix(x_raw, gene_names_raw, gene_names_model):
    assert len(gene_names_raw) == len(set(gene_names_raw))
    assert len(gene_names_model) == len(set(gene_names_model))
    assert len(gene_names_raw) == x_raw.shape[1]
    assert np.isin(gene_names_raw, gene_names_model).sum() == x_raw.shape[1]
    # For fast column-wise slicing matrix has to be in csc format
    assert isinstance(x_raw, csc_matrix)
    # skip zero filling missing genes if all genes are present
    if len(gene_names_raw) == len(gene_names_model):
        return x_raw.tocsr().astype('f4')
    gene_names_raw, gene_names_model = np.array(gene_names_raw), np.array(gene_names_model)
    row, col = np.empty(x_raw.nnz, dtype='i8'), np.empty(x_raw.nnz, dtype='i8')
    data = np.empty(x_raw.nnz, dtype='f4')

    ctr = 0    
    for i, gene in enumerate(gene_names_model):
        if gene in gene_names_raw:
            gene_idx = int(np.where(gene == gene_names_raw)[0])
            x_col = x_raw[:, gene_idx]
            idxs_nnz = x_col.indices.tolist()
            n_nnz = len(idxs_nnz)
            col[ctr:ctr+n_nnz] = i
            row[ctr:ctr+n_nnz] = idxs_nnz
            data[ctr:ctr+n_nnz] = x_col.data
            ctr += n_nnz

    return csr_matrix(
        (data, (row, col)),
        shape=(x_raw.shape[0], len(gene_names_model)),
        dtype='f4'
    )


In [14]:
from typing import List

import h5py
from anndata.experimental import read_elem


# columns to keep for the preprocessed data
COLUMNS = [
    'assay', 'cell_type', 'development_stage', 'disease', 'donor_id', 
    'is_primary_data', 'sex', 'suspension_type', 'tissue'
]


def preprocess_dataset(dataset_path: str, columns: List[str], cell_type_column: str):
    protein_coding_genes = pd.read_parquet('features.parquet')
    with h5py.File(dataset_path) as f:
        var = read_elem(f['var'])
        obs = read_elem(f['obs'])
        genes_to_use = np.isin(var.index.to_numpy(), protein_coding_genes.index.to_numpy())
        try:
            X = read_elem(f['raw']['X']).astype('f4').tocsc()[:, genes_to_use].copy()
        except KeyError:
            # if raw doesn't exist -> use .X instead
            # according to CELLxGENE schema
            # https://github.com/chanzuckerberg/single-cell-curation/blob/main/schema/3.0.0/schema.md#x-matrix-layers
            X = read_elem(f['X']).astype('f4').tocsc()[:, genes_to_use].copy()
        var = var.iloc[genes_to_use]

    # align feature spaces across datasets
    X = streamline_count_matrix(X, var.index.tolist(), protein_coding_genes.index.tolist())
    # subselect to desired obs columns
    obs = (
        obs[columns + [cell_type_column]]
        .copy()
        .rename(columns={cell_type_column: 'cell_type_author'})
    )

    return anndata.AnnData(X=X, obs=obs, var=protein_coding_genes)


In [15]:
for datasets, cell_type_column in dataset_collections:
    for dataset in datasets:
        print(f'Processing {dataset}')
        dataset_path = join(DOWNLOAD_PATH, dataset + '.h5ad')
        save_path = join(DOWNLOAD_PATH, dataset.removesuffix('.h5ad') + '_processed.h5ad')
        if not os.path.isfile(save_path):
            preprocess_dataset(dataset_path, COLUMNS, cell_type_column).write(save_path, compression='lzf')


Processing b0e547f0-462b-4f81-b31b-5b0a5d96f537
Processing ebc2e1ff-c8f9-466a-acf4-9d291afaf8b3
Processing ed5d841d-6346-47d4-ab2f-7119ad7e3a35
Processing de2c780c-1747-40bd-9ccf-9588ec186cee
Processing 21d3e683-80a4-4d9b-bc89-ebb2df513dde
Processing 30cd5311-6c09-46c9-94f1-71fe4b91813c
Processing 218acb0f-9f2f-4f76-b90b-15a4b7c7f629
Processing 96a3f64b-0ee9-40d8-91e9-813ce38261c9
Processing 01ad3cd7-3929-4654-84c0-6db05bd5fd59
Processing c2a461b1-0c15-4047-9fcb-1f966fe55100
Processing 3faad104-2ab8-4434-816d-474d8d2641db
Processing 2a498ace-872a-4935-984b-1afa70fd9886
